In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import pandas as pd

from torch.utils.data import DataLoader, random_split

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("All libraries imported successfully!")

In [2]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print("Training Images:", len(train_dataset))
print("Testing Images:", len(test_dataset))
print("CIFAR-10 Dataset Loaded Successfully!")

In [3]:
# Split training dataset

generator = torch.Generator().manual_seed(42)

train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=generator
)

print("Training Images:", len(train_subset))
print("Validation Images:", len(val_subset))
print("Testing Images:", len(test_dataset))

print("Dataset splitting completed successfully!")

In [4]:
# Set batch size
BATCH_SIZE = 64

# Training DataLoader
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Validation DataLoader
val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Testing DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

print("DataLoaders created successfully!")

In [5]:
class WidthCNN(nn.Module):

    def __init__(self, filters1, filters2):
        super(WidthCNN, self).__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(3, filters1, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(filters1, filters2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(filters2 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


# Create three width models

width_models = {
    "Width 1": WidthCNN(16, 32),
    "Width 2": WidthCNN(32, 64),
    "Width 3": WidthCNN(64, 128)
}

for name, model in width_models.items():
    print(name)
    print(model)
    print("-" * 50)

print("Three CNN Width Models created successfully!")

In [6]:
# Calculate trainable parameters

width_results = []

for name, model in width_models.items():

    total_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    width_results.append({
        "Model": name,
        "Trainable Parameters": total_params
    })

    print(name, ":", total_params, "parameters")

print("\nWidth parameter calculation completed!")

In [7]:
def train_model(model, train_loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


print("CNN Training function created successfully!")

In [8]:
def evaluate_model(model, data_loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in data_loader:

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(data_loader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


print("CNN Validation function created successfully!")

In [9]:
# Training configuration

EPOCHS = 10
criterion = nn.CrossEntropyLoss()

width_histories = {}

# Train all three width models

for name, model in width_models.items():

    print(f"\nTraining {name}")
    print("=" * 40)

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(EPOCHS):

        train_loss, train_acc = train_model(
            model, train_loader, criterion, optimizer
        )

        val_loss, val_acc = evaluate_model(
            model, val_loader, criterion
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} "
            f"Val Accuracy: {val_acc:.2f}%"
        )

    width_histories[name] = history

    print(f"{name} Training Completed!")

print("\nAll Width Models Training Completed!")

In [10]:
width_comparison = []

for name, history in width_histories.items():

    params = sum(
        p.numel() for p in width_models[name].parameters()
        if p.requires_grad
    )

    width_comparison.append({
        "Model": name,
        "Validation Accuracy (%)": round(
            history["val_accuracy"][-1], 2
        ),
        "Training Loss": round(
            history["train_loss"][-1], 4
        ),
        "Trainable Parameters": params
    })

width_df = pd.DataFrame(width_comparison)

print("WIDTH MODELS PERFORMANCE")
display(width_df)

In [11]:
plt.figure(figsize=(8, 5))

plt.bar(
    width_df["Model"],
    width_df["Validation Accuracy (%)"]
)

plt.title("Effect of Network Width on Validation Accuracy")
plt.xlabel("Network Width")
plt.ylabel("Validation Accuracy (%)")
plt.grid(axis="y")
plt.show()

In [12]:
plt.figure(figsize=(8, 5))

plt.bar(
    width_df["Model"],
    width_df["Training Loss"]
)

plt.title("Effect of Network Width on Training Loss")
plt.xlabel("Network Width")
plt.ylabel("Training Loss")
plt.grid(axis="y")
plt.show()

In [13]:
class DepthCNN(nn.Module):

    def __init__(self, channels):
        super(DepthCNN, self).__init__()

        layers = []
        in_channels = 3

        for out_channels in channels:

            layers.append(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=3,
                    padding=1
                )
            )

            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2, 2))

            in_channels = out_channels

        self.conv_layers = nn.Sequential(*layers)

        final_size = 32 // (2 ** len(channels))

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                channels[-1] * final_size * final_size,
                256
            ),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


# Create three depth models

depth_models = {
    "Depth 1": DepthCNN([32, 64]),
    "Depth 2": DepthCNN([32, 64, 128]),
    "Depth 3": DepthCNN([32, 64, 128, 256])
}

for name, model in depth_models.items():
    print(name)
    print(model)
    print("-" * 50)

print("Three CNN Depth Models created successfully!")

In [14]:
depth_results = []

for name, model in depth_models.items():

    total_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    depth_results.append({
        "Model": name,
        "Trainable Parameters": total_params
    })

    print(name, ":", total_params, "parameters")

print("\nDepth parameter calculation completed!")

In [15]:
EPOCHS = 10

depth_histories = {}

for name, model in depth_models.items():

    print(f"\nTraining {name}")
    print("=" * 40)

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(EPOCHS):

        train_loss, train_acc = train_model(
            model, train_loader, criterion, optimizer
        )

        val_loss, val_acc = evaluate_model(
            model, val_loader, criterion
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} "
            f"Val Accuracy: {val_acc:.2f}%"
        )

    depth_histories[name] = history

    print(f"{name} Training Completed!")

print("\nAll Depth Models Training Completed!")

In [16]:
depth_comparison = []

for name, history in depth_histories.items():

    params = sum(
        p.numel()
        for p in depth_models[name].parameters()
        if p.requires_grad
    )

    depth_comparison.append({
        "Model": name,
        "Validation Accuracy (%)": round(history["val_accuracy"][-1], 2),
        "Training Loss": round(history["train_loss"][-1], 4),
        "Trainable Parameters": params
    })

depth_df = pd.DataFrame(depth_comparison)

print("DEPTH MODELS PERFORMANCE")
display(depth_df)

In [17]:
plt.figure(figsize=(8, 5))

plt.bar(
    depth_df["Model"],
    depth_df["Validation Accuracy (%)"]
)

plt.title("Effect of Network Depth on Validation Accuracy")
plt.xlabel("Network Depth")
plt.ylabel("Validation Accuracy (%)")
plt.grid(axis="y")
plt.show()

In [18]:
plt.figure(figsize=(8, 5))

plt.bar(
    depth_df["Model"],
    depth_df["Training Loss"]
)

plt.title("Effect of Network Depth on Training Loss")
plt.xlabel("Network Depth")
plt.ylabel("Training Loss")
plt.grid(axis="y")
plt.show()

In [19]:
plt.figure(figsize=(8, 5))

plt.bar(
    depth_df["Model"],
    depth_df["Trainable Parameters"]
)

plt.title("Effect of Network Depth on Model Complexity")
plt.xlabel("Network Depth")
plt.ylabel("Trainable Parameters")
plt.grid(axis="y")
plt.show()

In [20]:
# Combine Width and Depth results

all_results = []

# Add Width models
for name, history in width_histories.items():

    params = sum(
        p.numel()
        for p in width_models[name].parameters()
        if p.requires_grad
    )

    all_results.append({
        "Model": name,
        "Type": "Width",
        "Validation Accuracy (%)": round(
            history["val_accuracy"][-1], 2
        ),
        "Training Loss": round(
            history["train_loss"][-1], 4
        ),
        "Trainable Parameters": params
    })


# Add Depth models
for name, history in depth_histories.items():

    params = sum(
        p.numel()
        for p in depth_models[name].parameters()
        if p.requires_grad
    )

    all_results.append({
        "Model": name,
        "Type": "Depth",
        "Validation Accuracy (%)": round(
            history["val_accuracy"][-1], 2
        ),
        "Training Loss": round(
            history["train_loss"][-1], 4
        ),
        "Trainable Parameters": params
    })


final_df = pd.DataFrame(all_results)

print("EXERCISE 04 - FINAL MODEL COMPARISON")
display(final_df)

In [21]:
plt.figure(figsize=(10, 5))

plt.bar(
    final_df["Model"],
    final_df["Validation Accuracy (%)"]
)

plt.title("Validation Accuracy Comparison - All Six CNN Models")
plt.xlabel("CNN Models")
plt.ylabel("Validation Accuracy (%)")
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.show()

In [22]:
plt.figure(figsize=(10, 5))

plt.bar(
    final_df["Model"],
    final_df["Trainable Parameters"]
)

plt.title("Computational Complexity - All Six CNN Models")
plt.xlabel("CNN Models")
plt.ylabel("Trainable Parameters")
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.show()

In [23]:
print("==========================================")
print("EXERCISE 04 - FINAL RESULTS SUMMARY")
print("==========================================")

print("Dataset: CIFAR-10")
print("Training Images: 45000")
print("Validation Images: 5000")
print("Testing Images: 10000")
print("Optimizer: Adam")
print("Learning Rate: 0.001")
print("Epochs: 10")
print("------------------------------------------")

print("WIDTH EXPERIMENT")
for name, history in width_histories.items():
    params = sum(
        p.numel() for p in width_models[name].parameters()
        if p.requires_grad
    )
    print(
        f"{name}: Validation Accuracy = "
        f"{history['val_accuracy'][-1]:.2f}%, "
        f"Parameters = {params}"
    )

print("------------------------------------------")

print("DEPTH EXPERIMENT")
for name, history in depth_histories.items():
    params = sum(
        p.numel() for p in depth_models[name].parameters()
        if p.requires_grad
    )
    print(
        f"{name}: Validation Accuracy = "
        f"{history['val_accuracy'][-1]:.2f}%, "
        f"Parameters = {params}"
    )

print("------------------------------------------")
print("Total CNN Models Evaluated: 6")
print("Exercise 04 Completed!")